In [1]:
## Naive Bayes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab') # Added as per error message
nltk.download('stopwords')
from nltk.tokenize import word_tokenize


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tungtran/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tungtran/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
# read data

df = pd.read_csv('emails.csv')
df.head(n=10)

,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0
5,Email 6,4,5,1,4,2,3,45,1,0,...,0,0,0,0,0,0,0,0,0,1
6,Email 7,5,3,1,3,2,1,37,0,0,...,0,0,0,0,0,0,0,0,0,0
7,Email 8,0,2,2,3,1,2,21,6,0,...,0,0,0,0,0,0,0,1,0,1
8,Email 9,2,2,3,0,0,1,18,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Email 10,4,4,35,0,1,0,49,1,16,...,0,0,0,0,0,0,0,0,0,0


In [4]:
from sklearn.model_selection import train_test_split

X = df.iloc[:, 1:-1]
Y = df['Prediction']

Y.head(n=10)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=1)

In [5]:
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('wordnet', quiet=True)

def text_preprocessing(mess):
  """
    Takes in a string of text, then performs the following:
    1. Remove all punctuation
    2. Remove all stopwords
    3. Returns a list of the cleaned text
    4. Lemmatization
  """
  lemmatizer = WordNetLemmatizer()
  STOPWORDS = stopwords.words('english') + ['u', 'ü', 'ur', '4', '2', 'im', 'dont', 'doin', 'ure']

  # Remove punctuation and join back to a string
  nopunc = ''.join([char for char in mess if char not in string.punctuation])

  # Tokenize the cleaned string into words
  words = nopunc.split()

  # Remove stopwords and lemmatize each word
  cleaned_words = [lemmatizer.lemmatize(word.lower(), pos='v') for word in words if word.lower() not in STOPWORDS]

  return ' '.join(cleaned_words)

In [ ]:
message = " This is a text message that probably should be removed after testing text Preprocessing"
message = text_preprocessing(message)
print(message)

text message probably remove test text preprocessing


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
vect_message = []
# Wrap the single message string in a list
# CountVectorizer.fit_transform() is expecting a list of text documents to process, you need to wrap your message variable in a list
vect_message = vect.fit_transform([message])

vect.get_feature_names_out()

array(['message', 'preprocessing', 'probably', 'remove', 'test', 'text'],
      dtype=object)

In [9]:
pd.DataFrame(vect_message.toarray(), columns=vect.get_feature_names_out())

,message,preprocessing,probably,remove,test,text
0,1,1,1,1,1,2


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [11]:
class MultinomialNB:
  def __init__(self, alpha=1):
      self.alpha = alpha

  def fit(self, X_train_input, y_train_input):
      # Ensure X_train is a dense NumPy array
      if hasattr(X_train_input, 'toarray'): # It's a sparse matrix
          X_train = X_train_input.toarray()
      elif isinstance(X_train_input, pd.DataFrame): # It's a pandas DataFrame
          X_train = X_train_input.to_numpy()
      else: # Assume it's already a dense NumPy array
          X_train = X_train_input

      # Ensure y_train is a NumPy array
      if isinstance(y_train_input, pd.Series):
          y_train = y_train_input.to_numpy()
      else:
          y_train = y_train_input

      m, n = X_train.shape
      self._classes = np.unique(y_train)
      n_classes = len(self._classes)
      # init: Prior & Likelihood
      self._priors = np.zeros(n_classes)
      self._likelihoods = np.zeros((n_classes, n))

      # Get Prior and Likelihood
      for idx, c in enumerate(self._classes):
          # Now X_train is a dense NumPy array, and y_train is a NumPy array
          # Direct boolean indexing works
          X_train_c = X_train[y_train == c, :]
          self._priors[idx] = X_train_c.shape[0] / m
          self._likelihoods[idx, :] = ((X_train_c.sum(axis=0)) + self.alpha) / (np.sum(X_train_c.sum(axis=0) + self.alpha))
      for idx, c in enumerate(self._classes):
        # print(self._likelihoods[idx, :])
        print(self._priors[idx])

  def predict(self, X_test):
      return [self._predict(x_test) for x_test in X_test]

  def _predict(self, x_test):
      # Calculate posterior for each class
      posteriors = []
      for idx, c in enumerate(self._classes):
          prior_c = np.log(self._priors[idx])
          likelihoods_c = self.calc_likelihood(self._likelihoods[idx,:], x_test)
          posteriors_c = np.sum(likelihoods_c) + prior_c
          posteriors.append(posteriors_c)

      return self._classes[np.argmax(posteriors)]

  def calc_likelihood(self, cls_likeli, x_test):
      return np.log(cls_likeli) * x_test

  def score(self, X_test, y_test):
      y_pred = self.predict(X_test)
      return np.sum(y_pred == y_test)/len(y_test)

In [12]:
classifier = MultinomialNB()
classifier.fit(X_train, y_train)
# classifier._predict(X_test[10])
x = X_test.iloc[30]
y = y_test.iloc[30]
print(x)
print(y)
print(x.shape)

0.7122969837587007
0.2877030162412993
the               4
to                2
ect               1
and               3
for               2
                 ..
infrastructure    0
military          0
allowing          0
ff                0
dry               0
Name: 4110, Length: 3000, dtype: int64
1
(3000,)


In [13]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predicting test results
y_pred = classifier.predict(X_test.to_numpy())
ytest = np.array(y_test)

# f1_score(ytest, y_pred, average='weighted')
print(classification_report(ytest, y_pred))
print(confusion_matrix(ytest, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.94      0.96       909
           1       0.88      0.96      0.92       384

    accuracy                           0.95      1293
   macro avg       0.93      0.95      0.94      1293
weighted avg       0.95      0.95      0.95      1293

[[857  52]
 [ 16 368]]


In [14]:
# Re-initialize a CountVectorizer and set its vocabulary from the training data columns.
# This ensures that when we transform new text, it produces a vector with the correct 3000 features.
from sklearn.feature_extraction.text import CountVectorizer

# Get the vocabulary from the training data columns
model_vocabulary = list(X.columns)

# Create a new CountVectorizer and set its vocabulary.
# This is crucial for matching the feature space of the trained model.
new_message_vectorizer = CountVectorizer(vocabulary=model_vocabulary)

print(f"Vectorizer vocabulary size: {len(new_message_vectorizer.vocabulary)}")
print(f"First 10 words in vocabulary: {new_message_vectorizer.vocabulary[:10]}")

Vectorizer vocabulary size: 3000
First 10 words in vocabulary: ['the', 'to', 'ect', 'and', 'for', 'of', 'a', 'you', 'hou', 'in']


In [15]:
raw_message = """URGENT! Your banking profile has been flagged due to unauthorized login attempts. Click here immediately to verify your identity and prevent account suspension: http://secure-bank-login-update.com. Failure to comply within 24 hours will result in permanent lockout."
    """
print(f"Original message: {raw_message}")


Original message: URGENT! Your banking profile has been flagged due to unauthorized login attempts. Click here immediately to verify your identity and prevent account suspension: http://secure-bank-login-update.com. Failure to comply within 24 hours will result in permanent lockout."
    


In [16]:
# Step 2: Preprocess the new message using the defined `text_preprocessing` function
cleaned_message = text_preprocessing(raw_message)
print(f"Cleaned message: {cleaned_message}")

Cleaned message: urgent bank profile flag due unauthorized login attempt click immediately verify identity prevent account suspension httpsecurebankloginupdatecom failure comply within 24 hours result permanent lockout


In [17]:
# Step 3: Vectorize the cleaned message using the `new_message_vectorizer`
# We wrap the cleaned message in a list because `transform` expects an iterable of documents.
vectorized_input = new_message_vectorizer.transform([cleaned_message])

# Display the shape of the vectorized input (should be 1 row, 3000 columns)
print(f"Shape of vectorized input: {vectorized_input.shape}")

# Convert to a dense numpy array for prediction (the `predict` method expects an iterable of 1D arrays)
message_for_prediction = vectorized_input.toarray()
print(f"First 10 features of the message: {message_for_prediction[0][:10]}")

Shape of vectorized input: (1, 3000)
First 10 features of the message: [0 0 0 0 0 0 0 0 0 0]


Finally, we can use the trained `classifier` to make a prediction on this prepared input.

In [18]:
# Step 4: Make a prediction
prediction = classifier.predict(message_for_prediction)

# The prediction will be a list, so get the first element.
predicted_class = prediction[0]

print(f"Predicted class for the new message: {predicted_class}")

# Assuming 0 for non-spam and 1 for spam (based on context of emails.csv 'Prediction' column)
class_label = "Spam" if predicted_class == 1 else "Not Spam"
print(f"The message is classified as: {class_label}")

Predicted class for the new message: 1
The message is classified as: Spam


In [19]:
def preprocess_and_vectorize_message(raw_text_message):
  """
  Preprocesses a raw text message and vectorizes it using the trained vocabulary.

  Args:
    raw_text_message (str): The raw text message to preprocess.

  Returns:
    numpy.ndarray: A dense NumPy array representing the vectorized message,
                   ready for prediction by the classifier.
  """
  vectorized_input = new_message_vectorizer.transform([cleaned_message])
  message_for_prediction = vectorized_input.toarray()

  return message_for_prediction

print("Function `preprocess_and_vectorize_message` defined.")

Function `preprocess_and_vectorize_message` defined.


In [20]:
processed_message_for_pred = preprocess_and_vectorize_message(raw_message)
print(f"Shape of processed message for prediction: {processed_message_for_pred.shape}")

prediction = classifier.predict(processed_message_for_pred)
predicted_class = prediction[0]

class_label = "Spam" if predicted_class == 1 else "Not Spam"

print(f"Predicted class for the new message (using function): {predicted_class}")
print(f"The message is classified as (using function): {class_label}")

Shape of processed message for prediction: (1, 3000)
Predicted class for the new message (using function): 1
The message is classified as (using function): Spam


In [21]:
import joblib

# Define filenames for saving
model_filename = 'multinomial_nb_classifier.joblib'
vectorizer_filename = 'count_vectorizer.joblib'

# Save the trained classifier
joblib.dump(classifier, model_filename)
print(f"Classifier saved to {model_filename}")

# Save the CountVectorizer
joblib.dump(new_message_vectorizer, vectorizer_filename)
print(f"Vectorizer saved to {vectorizer_filename}")

Classifier saved to multinomial_nb_classifier.joblib
Vectorizer saved to count_vectorizer.joblib


In [22]:
loaded_classifier = joblib.load(model_filename)
loaded_vectorizer = joblib.load(vectorizer_filename)

print(f"Classifier loaded from {model_filename}")
print(f"Vectorizer loaded from {vectorizer_filename}")

new_test_message = "Congratulations! You've won a free vacation. Click here to claim your prize!"
def independent_preprocess_and_vectorize(raw_text, vectorizer_obj):
    cleaned_text = text_preprocessing(raw_text)
    vectorized_text = vectorizer_obj.transform([cleaned_text]).toarray()
    return vectorized_text

processed_new_message = independent_preprocess_and_vectorize(new_test_message, loaded_vectorizer)

prediction_loaded = loaded_classifier.predict(processed_new_message)
predicted_class_loaded = prediction_loaded[0]

class_label_loaded = "Spam" if predicted_class_loaded == 1 else "Not Spam"

print(f"\nNew message: {new_test_message}")
print(f"Predicted class (using loaded model): {predicted_class_loaded}")
print(f"The new message is classified as (using loaded model): {class_label_loaded}")

Classifier loaded from multinomial_nb_classifier.joblib
Vectorizer loaded from count_vectorizer.joblib

New message: Congratulations! You've won a free vacation. Click here to claim your prize!
Predicted class (using loaded model): 1
The new message is classified as (using loaded model): Spam
